In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

import anndata
import scanpy as sc

from scipy.stats import mannwhitneyu

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Fail1', 'Delayed', 'Inter1', 'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/processed/rna_finegrain.h5ad')

In [ ]:
adata

In [ ]:
genes_path = '/u/project/cluo/chliu/Genomes/IGVF_hg38/annotations/ensembl_to_symbol.tsv'
genes = pd.read_csv(genes_path, sep='\t')
gene_dict = dict(zip(genes['gene'], genes['symbol']))

adata.var['ensg'] = adata.var.index
adata.var['symbol'] = [gene_dict.get(x) for x in adata.var['ensg']]
adata.var.index = adata.var['symbol']
adata = adata[:,~adata.var.index.duplicated()]

In [ ]:
ips_clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/label_transfer/snm3c_ips_subtype.csv'
ips_clusters = pd.read_csv(ips_clusters_path, sep='\t', header=0, index_col=0)
ips_clusters.shape

In [ ]:
ips_clusters['assay'].value_counts()

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['pred_coarse']
adata.obs['ips_subtype'] = ips_clusters.reindex(adata.obs.index)['ips_subtype']
ips_adata = adata[~adata.obs['ips_subtype'].isna()]
ips_adata.shape

In [ ]:
ips_adata.obs.head()

In [ ]:
tmp_meta = ips_adata.obs.copy()
tmp_meta.shape

In [ ]:
donors = ['C29', 'C37', 'C38', 'C39']
tmp_results = []

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
x_lab = 'ips_subtype'
y_lab = 'value'
hue = 'variable'

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    donor_meta = tmp_meta[tmp_meta['line'] == d]
    cc_plot = donor_meta.melt(id_vars=['ips_subtype'], value_vars=['S.Score', 'G2M.Score'])
    
    sns.boxplot(data=cc_plot, x=x_lab, y=y_lab, hue=hue, ax=ax, showfliers=False, palette='Dark2')
    ax.set_xlabel('', fontsize=16)
    ax.set_ylabel('', fontsize=16)
    ax.set_title(f'{d}', fontsize=20)

    ax.tick_params(axis='x', labelsize=12, labelrotation=45)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    cc_plot['donor'] = d
    tmp_results.append(cc_plot)
    
plt.subplots_adjust(wspace=0.8, hspace=0.8)

In [ ]:
donors = ['C29', 'C37', 'C38', 'C39']
tmp_results = []

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
x_lab = 'variable'
y_lab = 'value'
hue = 'ips_subtype'

for i, ax in enumerate(axes.flatten()):
    d = donors[i]
    donor_meta = tmp_meta[tmp_meta['line'] == d]
    cc_plot = donor_meta.melt(id_vars=['ips_subtype'], value_vars=['S.Score', 'G2M.Score'])
    
    sns.boxplot(data=cc_plot, x=x_lab, y=y_lab, hue=hue, ax=ax, showfliers=False)
    ax.set_xlabel('', fontsize=16)
    ax.set_ylabel('', fontsize=16)
    ax.set_title(f'{d}', fontsize=20)

    ax.tick_params(axis='x', labelsize=12, labelrotation=45)
    sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))
    cc_plot['donor'] = d
    tmp_results.append(cc_plot)
    
plt.subplots_adjust(wspace=0.8, hspace=0.8)

In [ ]:
merged = pd.concat(tmp_results)
merged.shape

In [ ]:
merged.head()

In [ ]:
s_merged = merged[merged['variable'] == 'S.Score']
g2m_merged = merged[merged['variable'] == 'G2M.Score']

In [ ]:
tmp_main = s_merged[s_merged['ips_subtype'] == 'IPS_main']
tmp_alt = s_merged[s_merged['ips_subtype'] == 'IPS_alt']

a = tmp_main['value']
b = tmp_alt['value']
mannwhitneyu(a, b)

In [ ]:
tmp_main = g2m_merged[g2m_merged['ips_subtype'] == 'IPS_main']
tmp_alt = g2m_merged[g2m_merged['ips_subtype'] == 'IPS_alt']

a = tmp_main['value']
b = tmp_alt['value']
mannwhitneyu(a, b)

In [ ]:
indir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure5'
merged.to_csv(f'{indir}/ips_subtype_rna_cell_cycle.csv', sep='\t')

### get nuclei stratified by G2M/S score

In [ ]:
ips_adata.obs['ips_subtype'].value_counts()

In [ ]:
ips_adata.obs.groupby('ips_subtype')['Phase']

In [ ]:
ips_adata.obs['final_phase'] = [x.replace('G0/G1', 'G0') for x in ips_adata.obs['Phase']]

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/ips_subtype'
for (phase, subtype), group in ips_adata.obs.groupby(['final_phase', 'ips_subtype']):
    print(subtype)
    print(phase)
    group.to_csv(f'{outdir}/{subtype}_{phase}.csv', sep='\t')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/csv/ips_subtype'
ips_adata.obs.to_csv(f'{outdir}/merged_meta.csv', sep='\t')

In [ ]:
!date